# Geração de imagens — Opalescencia Studio (Colab)

Este notebook gera 10 amostras no estilo "MS Paint bonito" (1280x720) e salva como timestamps: `000s.png`, `006s.png`, ...

Instruções rápidas:
1. No Colab, certifique-se de usar um runtime com GPU (Runtime -> Change runtime type -> GPU).
2. Antes de rodar, defina a variável de ambiente `HF_TOKEN` com um token do Hugging Face que tenha acesso ao modelo (se necessário).
3. Rode as células na ordem.

In [ ]:
# 1) Instale dependências (execute no Colab)
!pip install --quiet accelerate diffusers transformers safetensors==0.4.0 pillow opencv-python
# Opcional: instale xformers se disponível para melhor desempenho
# !pip install -q xformers

In [ ]:
# 2) Importes e configuração (inclua seu token se necessário)
import os
from pathlib import Path
from PIL import Image
import shutil

OUTPUT_DIR = Path('/content/output')
UPSCALED_DIR = Path('/content/output_upscaled')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPSCALED_DIR.mkdir(parents=True, exist_ok=True)

# Se usar HuggingFace private models, defina HF_TOKEN antes de rodar (Colab: use os.environ)
# os.environ['HF_TOKEN'] = 'your_hf_token'
print('Output dir:', OUTPUT_DIR)
print('Upscaled dir:', UPSCALED_DIR)

In [ ]:
# Diagnostic quick check: Drive, HF token and existing outputs (run once)
import os
from pathlib import Path
print('Python runtime check')
print('Drive mounted:', Path('/content/drive').exists())
print('HF_TOKEN set in env:', bool(os.environ.get('HF_TOKEN')))
OUT=Path('/content/output')
UP=Path('/content/output_upscaled')
DRIVE_OUT=Path('/content/drive/MyDrive/Opalescencia_Studio_outputs')
for p in (OUT, UP, DRIVE_OUT):
    print(p, 'exists?', p.exists())
# Show up to 10 existing images if present
from IPython.display import display, Image
candidates = []
if DRIVE_OUT.exists():
    candidates = sorted([f for f in DRIVE_OUT.iterdir() if f.is_file()])
if not candidates and OUT.exists():
    candidates = sorted([f for f in OUT.iterdir() if f.is_file()])
if candidates:
    print('Showing up to 10 found images (preview)')
    for img in candidates[:10]:
        display(Image(str(img), width=320))
else:
    print('No images found in standard output folders.')

In [ ]:
# 3) Prompt template e lista de prompts (10 amostras para o primeiro minuto)
prompt_template = (
    "A bold, expressive digital painting in a polished MS Paint-inspired style, vibrant blue and indigo palette, simple shapes with soft gradients, mystical cosmic symbolism, subtle spiritual visuals, centered composition that harmonizes with a blue logo and a dark consciousness expansion theme. Make the look refined and clean, with a hand-drawn digital feel, not hyperrealistic, balancing playful simplicity with deep meaning."
)
negative_prompt = 'photorealistic, high detail, 3D rendering, realistic face, ultra realistic, cinematic, text, logo, watermark, photo'

prompts = [
    "Ilustração digital no estilo MS Paint bonito: paleta azul e anil, formas simples, gradientes suaves, símbolos místicos sutis, sensação de certeza e calma, composição central, traço manual digital, clean e elegante.",
    "Mão estilizada segurando uma luz azul, pinceladas digitais simples, atmosfera contemplativa, fundo azul escuro com estrelas suaves, presença simbólica discreta.",
    "Portal azul minimal com gradientes suaves e pontos brilhantes, composição central, sensação de transição interior, traço limpo.",
    "Figura estilizada em meditação, silhueta simples com aura azul e pequenos símbolos flutuantes, aparência pintada à mão digitalmente.",
    "Impressora simbólica abstrata imprimindo formas azuis (metáfora do subconsciente), traço simples, leve granulação artística.",
    "Quadro de visão estilizado com imagens simples e cor azul dominante, elementos simbólicos, atmosfera serena e inspiradora.",
    "Coração/centro emocional representado por um núcleo brilhante azul dentro de um corpo simples; linhas radiantes mostrando sensação e energia.",
    "Dois sentimentos em conflito ilustrados por duas formas opostas (uma fria, outra quente) com predominância azul quando a certeza vence; composição simbólica.",
    "Quarto abafado que se transforma: cena simples mostrando ajuste simbólico de temperatura para azul, metáfora visual clara.",
    "Livro aberto com pequenas estrelas saindo das páginas, tom azul profundo, sensação de certeza e manifestação."
]

# Combine template + prompts
prompts = [prompt_template + ' ' + p for p in prompts]
for i,p in enumerate(prompts):
    print(f'{i}:', p[:120].strip() + '...')

In [ ]:
# 4) Carregar modelo (exemplo com diffusers).
# ATENCAO: Coloque seu HF_TOKEN em os.environ['HF_TOKEN'] se o modelo exigir autenticação.
from diffusers import StableDiffusionPipeline
import torch

model_id = 'stabilityai/stable-diffusion-xl-base-1.0'

# Tente carregar o pipeline em fp16 na GPU (Colab com GPU)
try:
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
    pipe = pipe.to('cuda')
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass
    print('Modelo carregado:', model_id)
except Exception as e:
    print('Falha ao carregar modelo:', e)
    print('Verifique seu HF token ou troque para um modelo público compatível.')

In [ ]:
# 3.5) (Opcional) Montar Google Drive para salvar resultados automaticamente
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT = Path('/content/drive/MyDrive/Opalescencia_Studio_outputs')
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    print('Drive montado, output em:', DRIVE_OUTPUT)
except Exception as e:
    print('Google Drive não disponível ou não montado:', e)
    DRIVE_OUTPUT = None

In [ ]:
# 5) Parâmetros de geração (edite aqui para gerar lote completo)
# Gerar o lote completo para um vídeo (~8 minutos) por padrão
GENERATE_FULL_BATCH = False  # gerar N imagens (lote completo). False gera apenas prompts definidos
FULL_BATCH_SIZE = 10
WIDTH, HEIGHT = 1280, 720
SAVE_INTERVAL = 10  # salvar a cada X imagens se desejar

# Determine quantas imagens gerar
if GENERATE_FULL_BATCH:
    total_images = FULL_BATCH_SIZE
else:
    total_images = len(prompts)

from PIL import Image

# Geração em lote: gera `total_images` imagens, reciclando prompts se necessário
for idx in range(total_images):
    prompt = prompts[idx % len(prompts)]
    try:
        result = pipe(prompt, negative_prompt=negative_prompt, num_inference_steps=25, guidance_scale=7.5, width=WIDTH, height=HEIGHT)
        image = result.images[0]
    except Exception as e:
        print('Erro gerando imagem:', e)
        image = Image.new('RGB', (WIDTH, HEIGHT), (10, 30, 70))
    filename = OUTPUT_DIR / f"{idx*6:03d}s.png"
    image.save(filename)
    # Copy to Drive if available to persist outputs immediately
    try:
        if 'DRIVE_OUTPUT' in globals() and DRIVE_OUTPUT is not None:
            shutil.copy(str(filename), str(DRIVE_OUTPUT / filename.name))
            print('Copied to Drive:', DRIVE_OUTPUT / filename.name)
    except Exception as e:
        print('Warning copying to Drive:', e)
    if idx % SAVE_INTERVAL == 0:
        print('Saved:', filename)

In [ ]:
# 6) Upscale: tentativa de usar Real-ESRGAN se disponível; senão fallback Lanczos
USE_REAL_ESRGAN = True
TARGET_SIZE = (1920, 1080)
if USE_REAL_ESRGAN:
    try:
        !pip install --quiet realesrgan basicsr
        from realesrgan import RealESRGAN
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = RealESRGAN(device, scale=2)
        model.load_weights('RealESRGAN_x2.pth', download=True)
        for img_path in sorted(OUTPUT_DIR.iterdir()):
            img = Image.open(img_path).convert('RGB')
            sr = model.predict(img)
            out_path = UPSCALED_DIR / img_path.name
            sr.save(out_path)
            print('Upscaled:', out_path)
    except Exception as e:
        print('Real-ESRGAN falhou, usando Lanczos:', e)
        USE_REAL_ESRGAN = False

if not USE_REAL_ESRGAN:
    for img_path in sorted(OUTPUT_DIR.iterdir()):
        img = Image.open(img_path).convert('RGB')
        up = img.resize(TARGET_SIZE, resample=Image.LANCZOS)
        out_path = UPSCALED_DIR / img_path.name
        up.save(out_path)
        print('Upscaled (Lanczos):', out_path)

In [ ]:
# 7) Montagem automática com MoviePy (usa imagens em `UPSCALED_DIR`)
from moviepy.editor import ImageClip, concatenate_videoclips
from pathlib import Path

fps = 30
clips = []
image_files = sorted(Path(UPSCALED_DIR).iterdir())
for i, p in enumerate(image_files):
    clip = ImageClip(str(p)).with_duration(4)
    if i != 0:
        from moviepy.video.fx.CrossFadeIn import CrossFadeIn
        clip = clip.with_effects([CrossFadeIn(0.5)])
    if i != len(image_files) - 1:
        from moviepy.video.fx.CrossFadeOut import CrossFadeOut
        clip = clip.with_effects([CrossFadeOut(0.5)])
    clips.append(clip)
final = concatenate_videoclips(clips, method='compose', padding=-0.5)
out_video = Path('/content/final_video.mp4')
final.write_videofile(str(out_video), fps=fps, codec='libx264', audio=False, threads=4, preset='medium')
print('Final video:', out_video)

# 8) Upload automático para Google Drive (se montado)
if 'DRIVE_OUTPUT' in globals() and DRIVE_OUTPUT is not None:
    dest = DRIVE_OUTPUT / out_video.name
    shutil.copy(str(out_video), str(dest))
    print('Uploaded final video to Drive:', dest)